In [14]:
from statistics import logistic_kernel

with open('input.txt' , 'r', encoding='utf-8') as f:
    text = f.read()

In [15]:
print("length of dataset in characters: " , len(text))

length of dataset in characters:  1115394


In [16]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [17]:
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i:ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[ch] for ch in s]
decode = lambda l: ''.join([itos[ch] for ch in l])

print(encode('hello'))
print(decode(encode('hello')))

[46, 43, 50, 50, 53]
hello


In [18]:
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape , data.type)
print(data[:1000])

torch.Size([1115394]) <built-in method type of Tensor object at 0x7f6a44117750>
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 

In [19]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [20]:
block_size = 8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [21]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f'when input is {context} the target is {target}')

when input is tensor([18]) the target is 47
when input is tensor([18, 47]) the target is 56
when input is tensor([18, 47, 56]) the target is 57
when input is tensor([18, 47, 56, 57]) the target is 58
when input is tensor([18, 47, 56, 57, 58]) the target is 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target is 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target is 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target is 58


In [23]:
torch.manual_seed(1337)
batch_size = 4
block_size = 8

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data)-block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb , yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('------------------')

for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t+1]
        target = yb[b, t]
        print(f'when input is {context} the target is {target}')

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
------------------
when input is tensor([24]) the target is 43
when input is tensor([24, 43]) the target is 58
when input is tensor([24, 43, 58]) the target is 5
when input is tensor([24, 43, 58,  5]) the target is 57
when input is tensor([24, 43, 58,  5, 57]) the target is 1
when input is tensor([24, 43, 58,  5, 57,  1]) the target is 46
when input is tensor([24, 43, 58,  5, 57,  1, 46]) the target is 43
when input is tensor([24, 43, 58,  5, 57,  1, 46, 43]) the target is 39
when input is tensor([44]) the target is 53
when input is tensor([44, 53]) the target is 56
when input is tensor([44, 53, 56])

In [31]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self,vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        logits = self.token_embedding_table(idx)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T , C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits , loss

    def generate(self, idx , max_new_tokens):
        for _ in range(max_new_tokens):
            logits , loss = self(idx)
            logits = logits[:,-1,:]
            probs = F.softmax(logits , dim =-1)
            idx_next = torch.multinomial(probs , num_samples=1)
            idx = torch.cat((idx, idx_next), dim = 1)
        return idx


m = BigramLanguageModel(vocab_size)
logits, loss = m(xb,yb)
print(logits.shape)
print(loss)
idx = torch.zeros((1,1) , dtype=torch.long)
print(decode(m.generate(idx, max_new_tokens=100)[0].tolist()))

torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [32]:
optimizer = torch.optim.AdamW(m.parameters() , lr=1e-3)

In [37]:
batch_size = 32
for steps in range(10000):
    xb , yb = get_batch('train')

    logits , loss = m(xb , yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    print(loss.item())

2.458448886871338
2.435842514038086
2.5739567279815674
2.440695285797119
2.4353625774383545
2.3432626724243164
2.4174892902374268
2.42624568939209
2.4616968631744385
2.405534267425537
2.568842887878418
2.495751142501831
2.565951347351074
2.4873571395874023
2.4961118698120117
2.5236799716949463
2.626885414123535
2.5586469173431396
2.439403533935547
2.320131778717041
2.459834337234497
2.4067084789276123
2.4868032932281494
2.3585760593414307
2.320200204849243
2.4048430919647217
2.5162129402160645
2.5285933017730713
2.5082507133483887
2.466888666152954
2.463843584060669
2.3803882598876953
2.381807565689087
2.5213773250579834
2.5403871536254883
2.461794137954712
2.433934450149536
2.4059183597564697
2.453428268432617
2.5425705909729004
2.5444769859313965
2.5545506477355957
2.3874659538269043
2.4144163131713867
2.408301591873169
2.47629714012146
2.4655940532684326
2.437547206878662
2.4408323764801025
2.4162285327911377
2.444427013397217
2.438042402267456
2.515058755874634
2.420337200164795
2.

In [40]:
print(decode(m.generate(idx, max_new_tokens=700)[0].tolist()))



ATo inchyig y;
Y:
S:
RLYOn h cas re be I it? meswed,
Noun
Thou that alizathive ly

Premo Twe HAus hyon,
HERis taldr g d whmathe pr gurkeortwhin Whakil te d;
Beranu ad-priou y!
Buerstoo, engin, anifiterit p,
isthey thopis'd
Bu?
He thilere apre thior ane; ofar pls swhengity, dyoloyororhes
Cisthe is thirnon f bapsard bo whithatheam athirtousend d deeroleed! s win pehin, wouers, IORDORY:
Orkitonce

Ort ce
O:
ffit herom mem, f bourothalurerug wnrourer; a ho os thay a ke we be yod man te d lokngheheare te st: I,
She norofrean is:
And tapleard appulant ds the parexier ofl tor l.

A ousamamo s Bed:
aurso we our su'st shesobel IVERDe haru tlarof, in a I ft wn,
He, bads be the.
Am om w iofuruekerth,

